In [0]:
# ============================================
# Stage 4.2: Load DimTerritory with PySpark
# ============================================

from pyspark.sql import functions as F

# 1. 读取 Raw Zone 的 SalesTerritory 数据
raw_path = "abfss://raw@stadventureworksdw.dfs.core.windows.net/AdventureWorks2019/SalesTerritory/"
df_territory = spark.read.parquet(raw_path)

print(f"Raw row count: {df_territory.count()}")
df_territory.printSchema()
display(df_territory)

Raw row count: 10
root
 |-- TerritoryID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- CountryRegionCode: string (nullable = true)
 |-- Group: string (nullable = true)
 |-- SalesYTD: decimal(19,4) (nullable = true)
 |-- SalesLastYear: decimal(19,4) (nullable = true)
 |-- CostYTD: decimal(19,4) (nullable = true)
 |-- CostLastYear: decimal(19,4) (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)



TerritoryID,Name,CountryRegionCode,Group,SalesYTD,SalesLastYear,CostYTD,CostLastYear,rowguid,ModifiedDate
1,Northwest,US,North America,7887186.7882,3298694.4938,0.0000,0.0000,43689a10-e30b-497f-b0de-11de20267ff7,2008-04-30T00:00:00.000Z
2,Northeast,US,North America,2402176.8476,3607148.9371,0.0000,0.0000,00fb7309-96cc-49e2-8363-0a1ba72486f2,2008-04-30T00:00:00.000Z
3,Central,US,North America,3072175.1180,3205014.0767,0.0000,0.0000,df6e7fd8-1a8d-468c-b103-ed8addb452c1,2008-04-30T00:00:00.000Z
4,Southwest,US,North America,10510853.8739,5366575.7098,0.0000,0.0000,dc3e9ea0-7950-4431-9428-99dbcbc33865,2008-04-30T00:00:00.000Z
5,Southeast,US,North America,2538667.2515,3925071.4318,0.0000,0.0000,6dc4165a-5e4c-42d2-809d-4344e0ac75e7,2008-04-30T00:00:00.000Z
6,Canada,CA,North America,6771829.1376,5693988.8600,0.0000,0.0000,06b4af8a-1639-476e-9266-110461d66b00,2008-04-30T00:00:00.000Z
7,France,FR,Europe,4772398.3078,2396539.7601,0.0000,0.0000,bf806804-9b4c-4b07-9d19-706f2e689552,2008-04-30T00:00:00.000Z
8,Germany,DE,Europe,3805202.3478,1307949.7917,0.0000,0.0000,6d2450db-8159-414f-a917-e73ee91c38a9,2008-04-30T00:00:00.000Z
9,Australia,AU,Pacific,5977814.9154,2278548.9776,0.0000,0.0000,602e612e-dfe9-41d9-b894-27e489747885,2008-04-30T00:00:00.000Z
10,United Kingdom,GB,Europe,5012905.3656,1635823.3967,0.0000,0.0000,05fc7e1f-2dea-414e-9ecd-09d150516fb5,2008-04-30T00:00:00.000Z


In [0]:
# 2. 转换：只保留需要的字段，重命名为 DW 的字段
dim_territory = (df_territory
    .select(
        F.col("TerritoryID").cast("int").alias("TerritoryID"),
        F.col("Name").alias("TerritoryName"),
        F.col("CountryRegionCode").cast("string").alias("CountryRegionCode"),
        F.col("`Group`").alias("GroupName"),   # 注意：Group 是保留字，需要反引号
    )
    # 添加代理键：用 monotonically_increasing_id 生成
    .withColumn("TerritoryKey", F.monotonically_increasing_id() + 1)
    .select("TerritoryKey", "TerritoryID", "TerritoryName", "CountryRegionCode", "GroupName")
)

display(dim_territory)

TerritoryKey,TerritoryID,TerritoryName,CountryRegionCode,GroupName
1,1,Northwest,US,North America
2,2,Northeast,US,North America
3,3,Central,US,North America
4,4,Southwest,US,North America
5,5,Southeast,US,North America
6,6,Canada,CA,North America
7,7,France,FR,Europe
8,8,Germany,DE,Europe
9,9,Australia,AU,Pacific
10,10,United Kingdom,GB,Europe


In [0]:
# 3. 写入 Curated Zone (Delta Lake)
curated_path = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimTerritory/"

dim_territory.write \
    .format("delta") \
    .mode("overwrite") \
    .save(curated_path)

print(f"✅ DimTerritory written to {curated_path}")

✅ DimTerritory written to abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimTerritory/


In [0]:
# 4. 验证
df_check = spark.read.format("delta").load(curated_path)
print(f"Total rows: {df_check.count()}")
# 期望：10

df_check.show()

Total rows: 10
+------------+-----------+--------------+-----------------+-------------+
|TerritoryKey|TerritoryID| TerritoryName|CountryRegionCode|    GroupName|
+------------+-----------+--------------+-----------------+-------------+
|           1|          1|     Northwest|               US|North America|
|           2|          2|     Northeast|               US|North America|
|           3|          3|       Central|               US|North America|
|           4|          4|     Southwest|               US|North America|
|           5|          5|     Southeast|               US|North America|
|           6|          6|        Canada|               CA|North America|
|           7|          7|        France|               FR|       Europe|
|           8|          8|       Germany|               DE|       Europe|
|           9|          9|     Australia|               AU|      Pacific|
|          10|         10|United Kingdom|               GB|       Europe|
+------------+---------